In [1]:
import os
import mlflow

os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"

mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")

In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/ced16f5d8bce44b5a9c3e335b0f484b4', creation_time=1784571024655, experiment_id='9', last_update_time=1784571024655, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [3]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

# Imports
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import ADASYN
import mlflow.sklearn
import optuna

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv('dataset.csv').dropna()
df.shape

(36662, 2)

In [6]:
# Step 1: Load and clean data
df = pd.read_csv('dataset.csv').dropna()
df = df.dropna(subset=['category'])

# Step 2: Split FIRST on raw text to avoid data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Step 3: TF-IDF vectorizer - FIT ONLY on training data
ngram_range = (1, 2)  # Bigram
max_features = 10000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train = vectorizer.fit_transform(X_train_raw)  # Fit ONLY on training data
X_test = vectorizer.transform(X_test_raw)  # Transform test data (don't refit)

# Step 4: Apply ADASYN ONLY to training data to handle class imbalance
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)

# Test data stays untouched for fair evaluation
X_train = X_train_resampled
y_train = y_train_resampled

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type and experiment info
        mlflow.set_tag("mlflow.runName", f"{model_name}_ADASYN_TFIDF_Bigrams_NoLeakage")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.set_tag("data_handling", "no_leakage_fixed")
        mlflow.set_tag("model_type", "LogisticRegression")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", str(ngram_range))
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("imbalance_handling", "ADASYN_on_train_only")
        mlflow.log_param("C", model.C)
        mlflow.log_param("penalty", model.penalty)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report metrics
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")
        
        print(f"Run logged successfully for {model_name}")


# Step 5: Optuna objective function for Logistic Regression
def objective_logreg(trial):
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])

    model = LogisticRegression(C=C, penalty=penalty, solver='saga', max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


# Step 6: Run Optuna for Logistic Regression, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_logreg, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LogisticRegression(C=best_params['C'], penalty=best_params['penalty'], solver='saga', max_iter=1000, random_state=42)

    # Log the best model with MLflow
    log_mlflow("LogisticRegression", best_model, X_train, X_test, y_train, y_test)
    
    print(f"\nBest Optuna parameters: {best_params}")
    print(f"Best trial accuracy: {study.best_value:.4f}")

# Run the experiment for Logistic Regression
run_optuna_experiment()

[I 2026-07-21 06:18:48,935] A new study created in memory with name: no-name-4881af86-be81-44fa-949c-b7002a03138e
[I 2026-07-21 06:18:50,513] Trial 0 finished with value: 0.22501022773762444 and parameters: {'C': 0.0009065062977938499, 'penalty': 'l1'}. Best is trial 0 with value: 0.22501022773762444.
[I 2026-07-21 06:18:52,091] Trial 1 finished with value: 0.4301104595663439 and parameters: {'C': 0.0026525860875173174, 'penalty': 'l1'}. Best is trial 1 with value: 0.4301104595663439.
[I 2026-07-21 06:18:55,519] Trial 2 finished with value: 0.5497068048547661 and parameters: {'C': 0.006809550901619615, 'penalty': 'l2'}. Best is trial 2 with value: 0.5497068048547661.
[I 2026-07-21 06:18:56,615] Trial 3 finished with value: 0.4301104595663439 and parameters: {'C': 0.0006271076833366758, 'penalty': 'l1'}. Best is trial 2 with value: 0.5497068048547661.
[I 2026-07-21 06:19:00,357] Trial 4 finished with value: 0.4350197736260739 and parameters: {'C': 0.0011315873714144418, 'penalty': 'l2'}

Run logged successfully for LogisticRegression

Best Optuna parameters: {'C': 0.799306390597417, 'penalty': 'l1'}
Best trial accuracy: 0.8679
